# SEC 10-K Sentiment-Based Portfolio Construction
## Project 2 — Full Pipeline (Daily Data Version)

**Pipeline overview**
1. Install dependencies
2. Configuration
3. Imports
4. Load daily return data & lookup tables
5. Discover 10-K files
6. Extract & clean Item 7 (MD&A)
7. Sentiment scoring — **dictionary method** (Bing Liu lexicon)
8. Sentiment scoring — **LLM method** (HuggingFace RoBERTa)
9. Master DataFrame & summary statistics
10. Trading strategy & backtesting (daily data)
11. Save results

---
**Changes vs monthly version:**
- Returns loaded from `stocks_daily.csv` (CRSP daily format)
- Holding period starts the **day after** the filing date — no look-ahead bias
- Holding period ends exactly `HOLDING_DAYS` calendar days after filing
- Returns are **compounded** daily: `prod(1 + r_t) - 1`
- Market-adjusted returns computed by subtracting compounded benchmark (`sprtrn` or `ewretx`)
- Sharpe ratio annualised using the correct period length

In [ ]:
# Call the required modules

import os
import re
import warnings
from pathlib import Path
from typing import Optional

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## 1. Install Dependencies

In [2]:
# pip install -q transformers torch

## 2. Configuration

In [3]:
import os
print(os.getcwd())

# ── Path configuration ────────────────────────────────────────────────────────
PATH_DRIVE      = '/Users/alexandreviolleau/Library/CloudStorage/GoogleDrive-zouglagalex@gmail.com/My Drive/Cours/HEC/Lessons/3_Third_quarter/Data_Analysis_For_Finance/Dropbox'
BASE_PATH       = '/home/jovyan/persistent'
SEC_DATA_PATH   = f'{BASE_PATH}/10-X_C_2024'
LOOKUP_PATH     = f'{BASE_PATH}/lookup_tables'
DICT_PATH       = f'{BASE_PATH}/words_dictionary'
DAILY_DATA_PATH = f'{BASE_PATH}/stocks_daily.csv'  # <-- daily CRSP file

# ── Scope of the run ──────────────────────────────────────────────────────────
YEARS    = [2023, 2024]
QUARTERS = ['QTR1', 'QTR2', 'QTR3', 'QTR4']
CIK_FILTER  = None  # e.g. ['1048911', '1775734'] or None
TOP_N_FIRMS = 500

# ── Sentiment method ──────────────────────────────────────────────────────────
SENTIMENT_METHOD = 'both'  # 'dictionary', 'llm', or 'both'

# ── LLM configuration (HuggingFace, CPU-friendly) ────────────────────────────
LLM_BACKEND   = 'huggingface'
HF_MODEL      = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
LLM_MAX_CHARS = 512  # RoBERTa token limit

# ── Holding period ────────────────────────────────────────────────────────────
# Calendar days after filing date. Common choices:
#   21  ≈ 1 month  |  63 ≈ 1 quarter  |  126 ≈ 6 months  |  252 ≈ 1 year
HOLDING_DAYS = 63

# ── Quintile split ────────────────────────────────────────────────────────────
QUINTILE_N = 5

# ── Benchmark column in daily data ────────────────────────────────────────────
# 'sprtrn' = S&P 500 total return (market-adjusted alpha)
# 'ewretx' = equal-weighted CRSP index
# None     = raw returns only (no benchmark adjustment)
BENCHMARK_COL = 'sprtrn'

# ── CIK -> PERMNO lookup table config ─────────────────────────────────────────
# Adjust to match your actual lookup table key and column names
CIK_PERMNO_TABLE = 'gvkey_cik_lookup_table'
CIK_COL_LU       = 'cik'
PERMNO_COL_LU    = 'permno'

print('Configuration loaded.')
print(f'  Years        : {YEARS}')
print(f'  Holding days : {HOLDING_DAYS}')
print(f'  Sentiment    : {SENTIMENT_METHOD}')
print(f'  Benchmark    : {BENCHMARK_COL}')

/Users/alexandreviolleau/Documents/Code/Data-Analysis-Project
Configuration loaded.
  Years        : [2023, 2024]
  Holding days : 63
  Sentiment    : both
  Benchmark    : sprtrn


## 3. Imports

In [4]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 60)
print('Imports OK.')

Imports OK.


## 4. Load Daily Return Data & Lookup Tables

Expected columns in `stocks_daily.csv`:
`PERMNO, date, SHRCD, EXCHCD, SICCD, TICKER, PRC, VOL, RET, SHROUT, ewretx, sprtrn`

The daily DataFrame is loaded once here and reused in the backtesting section.

In [5]:
# ── Load daily returns ────────────────────────────────────────────────────────
print(f'Loading daily data from: {DAILY_DATA_PATH}')
daily_df = pd.read_csv(DAILY_DATA_PATH, low_memory=False)

# Normalise column names to lowercase
daily_df.columns = daily_df.columns.str.strip().str.lower()
daily_df['date'] = pd.to_datetime(daily_df['date'], errors='coerce')

# Convert RET to numeric — CRSP uses 'C'/'B' codes for corporate actions
daily_df['ret'] = pd.to_numeric(daily_df['ret'], errors='coerce')

# Optional: keep only common shares (shrcd 10/11) on major exchanges
# daily_df = daily_df[daily_df['shrcd'].isin([10, 11])]

daily_df = daily_df.sort_values(['permno', 'date']).reset_index(drop=True)

print(f'Loaded {len(daily_df):,} rows | '
      f'{daily_df["permno"].nunique():,} unique PERMNOs | '
      f'{daily_df["date"].min().date()} → {daily_df["date"].max().date()}')
daily_df.head(3)

Loading daily data from: /home/jovyan/persistent/stocks_daily.csv


FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/persistent/stocks_daily.csv'

In [ ]:
# ── Load lookup tables ────────────────────────────────────────────────────────
def load_lookup_tables(lookup_path: str) -> dict:
    """Load every CSV / Parquet file in the lookup folder into a dict."""
    tables = {}
    for fname in os.listdir(lookup_path):
        fpath = os.path.join(lookup_path, fname)
        key   = Path(fname).stem
        try:
            if fname.endswith('.csv'):
                tables[key] = pd.read_csv(fpath, low_memory=False)
            elif fname.endswith(('.parquet', '.pq')):
                tables[key] = pd.read_parquet(fpath)
        except Exception as e:
            print(f'  Warning: could not load {fname}: {e}')
    print(f'Loaded {len(tables)} lookup table(s): {list(tables.keys())}')
    return tables

lookup_tables = load_lookup_tables(LOOKUP_PATH)

# ── Build CIK → PERMNO mapping ────────────────────────────────────────────────
if CIK_PERMNO_TABLE in lookup_tables:
    lu = lookup_tables[CIK_PERMNO_TABLE].copy()
    lu.columns = lu.columns.str.strip().str.lower()
    cik_to_permno = (
        lu[[CIK_COL_LU, PERMNO_COL_LU]]
        .dropna()
        .astype(str)
        .drop_duplicates(subset=[CIK_COL_LU])
        .set_index(CIK_COL_LU)[PERMNO_COL_LU]
        .to_dict()
    )
    print(f'CIK→PERMNO mapping: {len(cik_to_permno):,} entries.')
else:
    cik_to_permno = {}
    print(f'WARNING: table "{CIK_PERMNO_TABLE}" not found in lookup_tables.')
    print('  PERMNO matching will be skipped and backtesting will not run.')
    print(f'  Available tables: {list(lookup_tables.keys())}')

In [ ]:
# ── Select top-N firms by market cap ──────────────────────────────────────────
def get_top_n_ciks(lookup_tables: dict, years: list, top_n: int) -> dict:
    MCAP_TABLE = 'crsp_mcap'
    CIK_COL    = 'cik'
    DATE_COL   = 'date'
    MCAP_COL   = 'mktcap'

    if MCAP_TABLE not in lookup_tables:
        print(f'Table "{MCAP_TABLE}" not found — using all CIKs.')
        return {yr: None for yr in years}

    df = lookup_tables[MCAP_TABLE].copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    result = {}
    for yr in years:
        dec    = pd.Timestamp(yr - 1, 12, 31)
        window = df[df[DATE_COL] <= dec]
        if window.empty:
            result[yr] = None
            continue
        latest = window[DATE_COL].max()
        snap   = window[window[DATE_COL] == latest]
        top    = snap.nlargest(top_n, MCAP_COL)[CIK_COL].astype(str).str.strip().tolist()
        result[yr] = set(top)
        print(f'  {yr}: {len(top)} firms selected (snapshot: {latest.date()})')
    return result

if TOP_N_FIRMS is not None:
    top_ciks_by_year = get_top_n_ciks(lookup_tables, YEARS, TOP_N_FIRMS)
else:
    top_ciks_by_year = {yr: None for yr in YEARS}
    print('TOP_N_FIRMS=None — skipping market-cap filter.')

## 5. Discover 10-K Files

In [ ]:
FNAME_RE = re.compile(r'^(\d{8})_10-K_edgar_data_(\d+)_.*\.txt$')

def discover_10k_files(
    sec_data_path: str,
    years: list,
    quarters: Optional[list],
    top_ciks_by_year: dict,
    cik_filter: Optional[list],
) -> pd.DataFrame:
    records     = []
    manual_ciks = set(str(c) for c in cik_filter) if cik_filter else None

    for year in years:
        year_path = os.path.join(sec_data_path, str(year))
        if not os.path.isdir(year_path):
            print(f'  WARNING: year folder not found: {year_path}')
            continue
        year_ciks      = top_ciks_by_year.get(year)
        available_qtrs = sorted(
            d for d in os.listdir(year_path)
            if os.path.isdir(os.path.join(year_path, d)) and d.startswith('QTR')
        )
        target_qtrs = quarters if quarters else available_qtrs

        for qtr in target_qtrs:
            qtr_path = os.path.join(year_path, qtr)
            if not os.path.isdir(qtr_path):
                continue
            for fname in os.listdir(qtr_path):
                m = FNAME_RE.match(fname)
                if not m:
                    continue
                filing_date, cik = m.group(1), m.group(2)
                if manual_ciks and cik not in manual_ciks:
                    continue
                if year_ciks is not None and cik not in year_ciks:
                    continue
                records.append({
                    'year'    : year,
                    'quarter' : qtr,
                    'cik'     : cik,
                    'date'    : pd.to_datetime(filing_date, format='%Y%m%d'),
                    'filename': fname,
                    'filepath': os.path.join(qtr_path, fname),
                })

    df = pd.DataFrame(records)
    print(f'Discovered {len(df)} 10-K files matching the filters.')
    if not df.empty:
        print(df.groupby(['year', 'quarter']).size().rename('files').to_string())
    return df


filing_index = discover_10k_files(
    SEC_DATA_PATH, YEARS, QUARTERS, top_ciks_by_year, CIK_FILTER
)
filing_index.head()

## 6. Extract & Clean Item 7 (MD&A)

In [ ]:
_TAG  = r'(?:<[^>]+>|\s)+'
_APOS = r"[\u2019\'\s]+"

ITEM7_RE = re.compile(
    rf'ITEM{_TAG}7\.?{_TAG}MANAGEMENT{_APOS}S{_TAG}DISCUSSION{_TAG}AND{_TAG}ANALYSIS',
    re.IGNORECASE,
)
ITEM8_RE = re.compile(
    rf'ITEM{_TAG}8\.?{_TAG}FINANCIAL{_TAG}STATEMENTS',
    re.IGNORECASE,
)

def extract_item7(text: str) -> Optional[str]:
    """Return raw Item 7 text, or None if the section is not found."""
    matches7 = list(ITEM7_RE.finditer(text))
    if not matches7:
        return None
    start = matches7[-1].start()  # last occurrence → skip ToC
    end   = next(
        (m.start() for m in ITEM8_RE.finditer(text) if m.start() > start),
        len(text),
    )
    return text[start:end]

def clean_text(text: str) -> str:
    """Strip HTML tags and collapse whitespace."""
    no_tags = re.sub(r'<[^>]+>', ' ', text)
    return re.sub(r'\s+', ' ', no_tags).strip()

def process_filings(filing_index: pd.DataFrame) -> pd.DataFrame:
    """Read files, extract and clean Item 7, return enriched DataFrame."""
    item7_texts, statuses = [], []
    for _, row in filing_index.iterrows():
        try:
            with open(row['filepath'], 'r', encoding='utf-8', errors='replace') as f:
                raw = f.read()
            section = extract_item7(raw)
            if section:
                item7_texts.append(clean_text(section))
                statuses.append('ok')
            else:
                item7_texts.append(None)
                statuses.append('item7_not_found')
        except Exception as e:
            item7_texts.append(None)
            statuses.append(f'error: {e}')

    result = filing_index.copy()
    result['item7_text'] = item7_texts
    result['status']     = statuses
    ok  = (result['status'] == 'ok').sum()
    bad = len(result) - ok
    print(f'Extraction complete: {ok} succeeded, {bad} failed.')
    return result


filings_df = process_filings(filing_index)
filings_df[['year', 'quarter', 'cik', 'date', 'status']].head(10)

## 7. Dictionary-Based Sentiment (Bing Liu Lexicon)

**Scores computed:**
- `n_pos` / `n_neg` — raw word counts
- `sent_net`   — (pos − neg) / total words
- `sent_ratio` — pos / (pos + neg)

In [ ]:
def load_word_lists(dict_path: str):
    def _read(fname):
        fpath = os.path.join(dict_path, fname)
        words = set()
        with open(fpath, 'r', encoding='latin-1') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith(';'):
                    words.add(line.lower())
        return words
    pos_words = _read('positive-words.txt')
    neg_words = _read('negative-words.txt')
    print(f'Lexicon loaded: {len(pos_words)} positive, {len(neg_words)} negative words.')
    return pos_words, neg_words

pos_words, neg_words = load_word_lists(DICT_PATH)
_WORD_RE = re.compile(r"[a-z']+")

def dict_sentiment(text: str, pos_words: set, neg_words: set) -> dict:
    if not text:
        return {'n_pos': 0, 'n_neg': 0, 'n_words': 0,
                'sent_net': np.nan, 'sent_ratio': np.nan}
    tokens     = _WORD_RE.findall(text.lower())
    n_words    = len(tokens)
    n_pos      = sum(1 for t in tokens if t in pos_words)
    n_neg      = sum(1 for t in tokens if t in neg_words)
    sent_net   = (n_pos - n_neg) / n_words if n_words else np.nan
    denom      = n_pos + n_neg
    sent_ratio = n_pos / denom if denom else np.nan
    return {'n_pos': n_pos, 'n_neg': n_neg, 'n_words': n_words,
            'sent_net': sent_net, 'sent_ratio': sent_ratio}

if SENTIMENT_METHOD in ('dictionary', 'both'):
    sent_records = filings_df['item7_text'].apply(
        lambda t: dict_sentiment(t, pos_words, neg_words)
    )
    sent_df = pd.DataFrame(sent_records.tolist())
    for col in sent_df.columns:
        filings_df[f'dict_{col}'] = sent_df[col].values
    print('Dictionary sentiment scores added.')
    filings_df[['cik', 'date', 'dict_sent_net', 'dict_sent_ratio']].dropna().head()

## 8. LLM-Based Sentiment (HuggingFace RoBERTa)

In [ ]:
if SENTIMENT_METHOD in ('llm', 'both'):
    from transformers import pipeline as hf_pipeline
    print(f'Loading HuggingFace model: {HF_MODEL} ...')
    hf_classifier = hf_pipeline(
        'text-classification',
        model=HF_MODEL,
        truncation=True,
        max_length=512,
    )
    print('Model loaded.')

def llm_sentiment_hf(text: str, max_chars: int) -> dict:
    """Score text with the HuggingFace RoBERTa classifier."""
    if not text:
        return {'llm_sentiment': None, 'llm_score': np.nan, 'llm_rationale': 'no text'}
    try:
        result    = hf_classifier(text[:max_chars])[0]
        label     = result['label'].lower()
        hf_score  = result['score']
        score_map = {'negative': 1 - hf_score, 'neutral': 0.5, 'positive': hf_score}
        return {
            'llm_sentiment': label,
            'llm_score'    : score_map.get(label, 0.5),
            'llm_rationale': f'HuggingFace: {label} ({hf_score:.3f})',
        }
    except Exception as e:
        return {'llm_sentiment': None, 'llm_score': np.nan, 'llm_rationale': str(e)}

def score_filings_llm(filings_df: pd.DataFrame, max_chars: int) -> pd.DataFrame:
    results = []
    total   = len(filings_df)
    for i, (_, row) in enumerate(filings_df.iterrows(), 1):
        if i % 50 == 0 or i == 1:
            print(f'  LLM scoring {i}/{total}...', end='\r')
        results.append(llm_sentiment_hf(row.get('item7_text'), max_chars))
    print(f'\n  Done scoring {total} filings.')
    df = filings_df.copy()
    df['llm_sentiment'] = [r['llm_sentiment'] for r in results]
    df['llm_score']     = [r['llm_score']     for r in results]
    df['llm_rationale'] = [r['llm_rationale'] for r in results]
    return df

if SENTIMENT_METHOD in ('llm', 'both'):
    filings_df = score_filings_llm(filings_df, LLM_MAX_CHARS)
    print(filings_df[['cik', 'date', 'llm_sentiment', 'llm_score']].dropna().head())

## 9. Master DataFrame & Summary Statistics

In [ ]:
# Drop failed extractions
master_df = filings_df[filings_df['status'] == 'ok'].copy()
master_df = master_df.drop(columns=['filepath', 'status'], errors='ignore')

# Map CIK → PERMNO (needed to join with daily_df)
master_df['permno'] = master_df['cik'].map(cik_to_permno)
n_matched = master_df['permno'].notna().sum()
print(f'Master DataFrame: {len(master_df)} filings, '
      f'{n_matched} matched to a PERMNO ({n_matched/len(master_df):.1%} match rate)')

master_df.describe(include='all').T

In [ ]:
# ── Choose score column ────────────────────────────────────────────────────────
if SENTIMENT_METHOD == 'dictionary':
    SCORE_COL = 'dict_sent_net'
elif SENTIMENT_METHOD == 'llm':
    SCORE_COL = 'llm_score'
else:  # 'both' — prefer LLM if available, else dictionary
    llm_ok    = 'llm_score' in master_df.columns and master_df['llm_score'].notna().any()
    SCORE_COL = 'llm_score' if llm_ok else 'dict_sent_net'

print(f'Score column: {SCORE_COL}')

# ── Sentiment distribution plots ──────────────────────────────────────────────
n_plots = 2 if SENTIMENT_METHOD == 'both' else 1
fig, axes = plt.subplots(1, n_plots, figsize=(12, 4))
if n_plots == 1:
    axes = [axes]

if SENTIMENT_METHOD in ('dictionary', 'both'):
    ax = axes[0]
    master_df['dict_sent_net'].dropna().hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Dictionary Sentiment (net score)')
    ax.set_xlabel('(pos − neg) / total words')
    ax.axvline(0, color='red', linestyle='--', linewidth=1)

if SENTIMENT_METHOD in ('llm', 'both'):
    ax = axes[-1]
    master_df['llm_score'].dropna().hist(bins=40, ax=ax, color='darkorange', edgecolor='white')
    ax.set_title('LLM Sentiment Score')
    ax.set_xlabel('Score (0 = negative, 1 = positive)')
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

## 10. Trading Strategy & Backtesting (Daily Data)

**How it works:**
1. For each quarter, rank firms by sentiment score into `QUINTILE_N` buckets.
2. **Long** the top quintile, **Short** the bottom quintile.
3. Each firm's holding period runs from the **day after its filing date** to `filing_date + HOLDING_DAYS` — this is firm-specific, which is a key advantage of daily data over monthly.
4. Returns are compounded daily: `∏(1 + r_t) − 1`.
5. If `BENCHMARK_COL` is set, market-adjusted return = stock return − benchmark return over the same window.

In [ ]:
def compute_holding_return(
    permno: str,
    filing_date: pd.Timestamp,
    daily_df: pd.DataFrame,
    holding_days: int,
    benchmark_col: Optional[str],
) -> dict:
    """
    Compound daily returns for one stock over its holding window.

    Window: (filing_date, filing_date + holding_days]
      - Start is exclusive  → day after filing, no look-ahead bias
      - End   is inclusive  → full holding_days calendar days

    Returns:
      raw_ret  : compounded total return
      adj_ret  : raw_ret minus compounded benchmark return (if benchmark_col set)
      n_days   : number of trading days in the window
    """
    start = filing_date
    end   = filing_date + pd.Timedelta(days=holding_days)

    stock_rets = daily_df[
        (daily_df['permno'].astype(str) == str(permno)) &
        (daily_df['date'] > start) &
        (daily_df['date'] <= end)
    ]['ret'].dropna()

    if stock_rets.empty:
        return {'raw_ret': np.nan, 'adj_ret': np.nan, 'n_days': 0}

    raw_ret = (1 + stock_rets).prod() - 1
    adj_ret = np.nan

    if benchmark_col and benchmark_col in daily_df.columns:
        bench_rets = daily_df[
            (daily_df['date'] > start) &
            (daily_df['date'] <= end)
        ][benchmark_col].dropna()
        if not bench_rets.empty:
            adj_ret = raw_ret - ((1 + bench_rets).prod() - 1)

    return {'raw_ret': raw_ret, 'adj_ret': adj_ret, 'n_days': len(stock_rets)}


print('compute_holding_return() defined.')

In [ ]:
def build_long_short_portfolio_daily(
    master_df: pd.DataFrame,
    daily_df: pd.DataFrame,
    score_col: str,
    holding_days: int,
    benchmark_col: Optional[str],
    quintile_n: int = 5,
) -> pd.DataFrame:
    """
    Build an equal-weight long/short quintile portfolio using daily returns.

    Each firm's holding period is firm-specific (starts the day after its
    own filing date), making full use of the daily data granularity.

    Returns a DataFrame: one row per filing-quarter period.
    """
    ret_key = 'adj_ret' if benchmark_col else 'raw_ret'

    # Keep only rows with PERMNO and valid score
    df = master_df.dropna(subset=[score_col, 'permno']).copy()
    df['period'] = df['date'].dt.to_period('Q')

    results = []

    for period, grp in df.groupby('period'):
        grp = grp.dropna(subset=[score_col]).copy()
        if len(grp) < quintile_n * 2:
            continue

        grp['quintile'] = pd.qcut(
            grp[score_col], quintile_n, labels=False, duplicates='drop'
        )

        long_grp  = grp[grp['quintile'] == quintile_n - 1]
        short_grp = grp[grp['quintile'] == 0]

        def ew_return(sub_grp):
            """Equal-weight average of compounded holding returns."""
            rets = []
            for _, row in sub_grp.iterrows():
                r = compute_holding_return(
                    row['permno'], row['date'],
                    daily_df, holding_days, benchmark_col
                )
                if not np.isnan(r[ret_key]):
                    rets.append(r[ret_key])
            return np.mean(rets) if rets else np.nan

        long_ret  = ew_return(long_grp)
        short_ret = ew_return(short_grp)
        ls_ret    = (long_ret - short_ret
                     if not (np.isnan(long_ret) or np.isnan(short_ret))
                     else np.nan)

        results.append({
            'period'   : str(period),
            'long_ret' : long_ret,
            'short_ret': short_ret,
            'ls_ret'   : ls_ret,
            'n_long'   : len(long_grp),
            'n_short'  : len(short_grp),
            'ret_type' : ret_key,
        })
        print(f'  {period}  long n={len(long_grp):3d}  short n={len(short_grp):3d}  '
              f'L={long_ret:.2%}  S={short_ret:.2%}  L/S={ls_ret:.2%}')

    return pd.DataFrame(results)


portfolio_df = build_long_short_portfolio_daily(
    master_df      = master_df,
    daily_df       = daily_df,
    score_col      = SCORE_COL,
    holding_days   = HOLDING_DAYS,
    benchmark_col  = BENCHMARK_COL,
    quintile_n     = QUINTILE_N,
)
print('\nPortfolio summary:')
print(portfolio_df)

In [ ]:
# ── Cumulative return plot ─────────────────────────────────────────────────────
if not portfolio_df.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    cum = (1 + portfolio_df.set_index('period')[['long_ret', 'short_ret', 'ls_ret']]).cumprod()
    cum.plot(ax=ax, marker='o', markersize=4)

    bench_label = f'market-adjusted vs {BENCHMARK_COL}' if BENCHMARK_COL else 'raw'
    ax.set_title(
        f'Long/Short Sentiment Strategy — Cumulative Returns\n'
        f'({bench_label}, {HOLDING_DAYS}-day hold, {SCORE_COL})'
    )
    ax.set_ylabel('Cumulative Return')
    ax.axhline(1, color='black', linewidth=0.8, linestyle='--')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # ── Summary statistics ────────────────────────────────────────────────────
    # Annualisation: each holding period = HOLDING_DAYS calendar days
    # Approximate trading days: HOLDING_DAYS * 252/365
    trading_days_per_period = HOLDING_DAYS * 252 / 365
    periods_per_year        = 252 / trading_days_per_period

    print('\n=== Strategy Summary ===')
    print(f'  Holding period : {HOLDING_DAYS} calendar days (~{trading_days_per_period:.0f} trading days)')
    print(f'  Return type    : {bench_label}')
    print(f'  Score column   : {SCORE_COL}')
    print()
    print(f'  {"Leg":<8} {"Ann.Ret":>10} {"Ann.Vol":>10} {"Sharpe":>8} {"n":>5}')
    print('  ' + '-' * 45)
    for col, label in [('long_ret', 'Long'), ('short_ret', 'Short'), ('ls_ret', 'L/S')]:
        s = portfolio_df[col].dropna()
        if s.empty:
            print(f'  {label:<8} no data')
            continue
        ann = (1 + s.mean()) ** periods_per_year - 1   # geometric annualisation
        vol = s.std() * np.sqrt(periods_per_year)      # annualised volatility
        sr  = ann / vol if vol else np.nan
        print(f'  {label:<8} {ann:>10.2%} {vol:>10.2%} {sr:>8.2f} {len(s):>5}')
else:
    print('Portfolio is empty.')
    print('Check that: (1) PERMNO mapping is populated, '
          '(2) daily_df date range covers your filing dates + holding period, '
          '(3) sentiment scores are non-null.')

## 11. Save Results

In [ ]:
OUTPUT_DIR = f'{BASE_PATH}/results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

master_path = os.path.join(OUTPUT_DIR, 'filings_sentiment.csv')
master_df.to_csv(master_path, index=False)
print(f'Saved: {master_path}')

if not portfolio_df.empty:
    port_path = os.path.join(OUTPUT_DIR, 'portfolio_returns_daily.csv')
    portfolio_df.to_csv(port_path, index=False)
    print(f'Saved: {port_path}')

print('All outputs saved.')